In [36]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, col

In [50]:
existing_spark = SparkSession.getActiveSession()

if existing_spark:
    print("Existing Spark session found. Stopping it...")
    existing_spark.stop()

In [37]:
spark = (
        SparkSession.builder
        .appName("Analytics App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    )

26/04/26 10:23:33 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [38]:
spark

In [48]:
## ROLLUP        -> Generates hierarchical subtotals from left to right, ending with a grand total.
## CUBE          -> Generates all possible combinations of grouping columns.
## GROUPING SETS -> Lets you specify custom groupings manually.

In [39]:
data = [
    ("India", "Karnataka", "Bangalore", 100),
    ("India", "Karnataka", "Mysore", 80),
    ("India", "Maharashtra", "Mumbai", 150),
    ("USA", "California", "LA", 200),
]

columns = ["country", "state", "city", "sales"]

In [40]:
df = spark.createDataFrame(data, columns)
df.show()

+-------+-----------+---------+-----+
|country|      state|     city|sales|
+-------+-----------+---------+-----+
|  India|  Karnataka|Bangalore|  100|
|  India|  Karnataka|   Mysore|   80|
|  India|Maharashtra|   Mumbai|  150|
|    USA| California|       LA|  200|
+-------+-----------+---------+-----+



## Rollup

In [41]:
df.rollup("country", "state", "city").agg(sum("sales").alias("total_sales")).orderBy(col("country").desc()).show()

[Stage 78:===================================================>    (11 + 1) / 12]

+-------+-----------+---------+-----------+
|country|      state|     city|total_sales|
+-------+-----------+---------+-----------+
|    USA|       NULL|     NULL|        200|
|    USA| California|     NULL|        200|
|    USA| California|       LA|        200|
|  India|       NULL|     NULL|        330|
|  India|  Karnataka|     NULL|        180|
|  India|  Karnataka|Bangalore|        100|
|  India|  Karnataka|   Mysore|         80|
|  India|Maharashtra|   Mumbai|        150|
|  India|Maharashtra|     NULL|        150|
|   NULL|       NULL|     NULL|        530|
+-------+-----------+---------+-----------+



In [42]:
df.rollup("state", "city").agg(sum("sales").alias("total_sales")).orderBy(col("state").desc()).show()

+-----------+---------+-----------+
|      state|     city|total_sales|
+-----------+---------+-----------+
|Maharashtra|   Mumbai|        150|
|Maharashtra|     NULL|        150|
|  Karnataka|     NULL|        180|
|  Karnataka|Bangalore|        100|
|  Karnataka|   Mysore|         80|
| California|       LA|        200|
| California|     NULL|        200|
|       NULL|     NULL|        530|
+-----------+---------+-----------+



## Cube

In [43]:
df.cube("country", "state", "city").agg(sum("sales").alias("total_sales")).orderBy("country", "state", "city").show(40)

+-------+-----------+---------+-----------+
|country|      state|     city|total_sales|
+-------+-----------+---------+-----------+
|   NULL|       NULL|     NULL|        530|
|   NULL|       NULL|Bangalore|        100|
|   NULL|       NULL|       LA|        200|
|   NULL|       NULL|   Mumbai|        150|
|   NULL|       NULL|   Mysore|         80|
|   NULL| California|     NULL|        200|
|   NULL| California|       LA|        200|
|   NULL|  Karnataka|     NULL|        180|
|   NULL|  Karnataka|Bangalore|        100|
|   NULL|  Karnataka|   Mysore|         80|
|   NULL|Maharashtra|     NULL|        150|
|   NULL|Maharashtra|   Mumbai|        150|
|  India|       NULL|     NULL|        330|
|  India|       NULL|Bangalore|        100|
|  India|       NULL|   Mumbai|        150|
|  India|       NULL|   Mysore|         80|
|  India|  Karnataka|     NULL|        180|
|  India|  Karnataka|Bangalore|        100|
|  India|  Karnataka|   Mysore|         80|
|  India|Maharashtra|     NULL| 

In [44]:
df.filter(col("country") == "USA").cube("country", "state", "city").agg(sum("sales").alias("total_sales")).orderBy("country", "state", "city").show(40)

+-------+----------+----+-----------+
|country|     state|city|total_sales|
+-------+----------+----+-----------+
|   NULL|      NULL|NULL|        200|
|   NULL|      NULL|  LA|        200|
|   NULL|California|NULL|        200|
|   NULL|California|  LA|        200|
|    USA|      NULL|NULL|        200|
|    USA|      NULL|  LA|        200|
|    USA|California|NULL|        200|
|    USA|California|  LA|        200|
+-------+----------+----+-----------+



## GroupSets

In [45]:
df.createOrReplaceTempView("sales")

In [46]:
result = spark.sql("""
SELECT
    country,
    state,
    SUM(sales) AS total_sales
FROM sales
GROUP BY GROUPING SETS (
    (country, state),
    (country),
    ()
)
""")

result.show()

+-------+-----------+-----------+
|country|      state|total_sales|
+-------+-----------+-----------+
|  India|  Karnataka|        180|
|   NULL|       NULL|        530|
|  India|       NULL|        330|
|  India|Maharashtra|        150|
|    USA| California|        200|
|    USA|       NULL|        200|
+-------+-----------+-----------+



In [47]:
result = spark.sql("""
SELECT
    country,
    state,
    city,
    SUM(sales) AS total_sales
FROM sales
GROUP BY GROUPING SETS (
    (country, state, city),
    (country, state),
    (country),
    ()
)
""")

result.show()

+-------+-----------+---------+-----------+
|country|      state|     city|total_sales|
+-------+-----------+---------+-----------+
|   NULL|       NULL|     NULL|        530|
|  India|       NULL|     NULL|        330|
|  India|  Karnataka|     NULL|        180|
|  India|  Karnataka|Bangalore|        100|
|  India|  Karnataka|   Mysore|         80|
|  India|Maharashtra|   Mumbai|        150|
|  India|Maharashtra|     NULL|        150|
|    USA|       NULL|     NULL|        200|
|    USA| California|     NULL|        200|
|    USA| California|       LA|        200|
+-------+-----------+---------+-----------+

